# Домашнее задание 14
Тема: эмбеддинги, индекс `FAISS`, оценка качества retrieval, обновление базы знаний и базовый mini-RAG.

## 2. Задание
### 2.3.1. Импорты, seed и среда

In [1]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn as sk
import faiss

import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

# Фиксация SEED
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Устройство
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
else:
    DEVICE = torch.device("cpu")
    torch.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Создание папок для артефактов
ARTIFACTS_DIR = Path("./artifacts")
FIGURES_DIR = ARTIFACTS_DIR / "figures"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
print(f"Seed: {SEED}")

PyTorch: 2.10.0
Device: cpu
Seed: 42


In [2]:
# Загрузка и подготовка дополнительных зависимостей
import subprocess
import sys
from typing import List, Dict, Tuple, Optional

def ensure_package(package_name: str, import_name: Optional[str] = None) -> None:
    """Пытается импортировать пакет и при необходимости установить его через pip."""
    target = import_name or package_name
    try:
        __import__(target)
    except Exception:
        print(f"Устанавливаем пакет: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

# Установка необходимых пакетов
ensure_package("faiss-cpu", "faiss")
ensure_package("sentence-transformers", "sentence_transformers")

try:
    import faiss
    FAISS_AVAILABLE = True
except Exception as e:
    FAISS_AVAILABLE = False
    print("FAISS недоступен, будет использован fallback:", repr(e))

print(f"FAISS доступен: {FAISS_AVAILABLE}")


FAISS доступен: True


#### 2.3.2. База знаний и первичный анализ

In [3]:
# Загружаем базу знаний: используем реальный датасет AG News из интернета
# Сохраняем датасет в папку data для воспроизводимости
print("Загружаем датасет AG News из интернета...")
from datasets import load_dataset
import pickle

# Создаем папку data если её нет
data_dir = Path("./data")
data_dir.mkdir(exist_ok=True)

try:
    # Загружаем датасет (первый раз может требовать интернета, потом кешируется)
    dataset = load_dataset("ag_news", split="train[:12]")
    
    # Сохраняем датасет на диск для воспроизводимости
    dataset.save_to_disk(str(data_dir / "ag_news_sample"))
    print(f"Датасет сохранен в: {data_dir / 'ag_news_sample'}")
    
    # Преобразуем загруженные новости в формат документов
    documents: List[Dict[str, str]] = []
    for idx, item in enumerate(dataset, 1):
        documents.append({
            "doc_id": str(idx),
            "title": f"News {idx} (Category: {dataset.features['label'].names[item['label']]})",
            "text": item['text']
        })
    
    print(f"Успешно загружено документов из AG News: {len(documents)}")
    
except Exception as e:
    print(f"Ошибка загрузки AG News, используем локальный датасет: {e}")
    # Fallback: если нет интернета, используем предзагруженные новости (реалистичные)
    documents: List[Dict[str, str]] = [
        {
            "doc_id": "1",
            "title": "Technology News 1",
            "text": "UK: The British government has announced new policies on technology regulation, aiming to create a framework for responsible AI development and oversight of emerging technologies.",
        },
        {
            "doc_id": "2",
            "title": "Technology News 2",
            "text": "USA: Tech companies in Silicon Valley continue to innovate rapidly. New startups are disrupting traditional industries with machine learning applications and cloud computing solutions.",
        },
        {
            "doc_id": "3",
            "title": "Technology News 3",
            "text": "Germany: European Union data protection authorities expressed concern about data privacy violations, emphasizing the importance of GDPR compliance across all member states.",
        },
        {
            "doc_id": "4",
            "title": "Technology News 4",
            "text": "Japan: Japanese manufacturers are investing heavily in robotics and automation technologies to address labor shortages and improve productivity in manufacturing sectors.",
        },
        {
            "doc_id": "5",
            "title": "Technology News 5",
            "text": "India: The Indian tech industry is experiencing rapid growth with increased focus on software development, business process outsourcing, and IT services exports globally.",
        },
        {
            "doc_id": "6",
            "title": "Technology News 6",
            "text": "Canada: Canadian universities and research institutions are collaborating on AI research projects, receiving government funding for developing cutting-edge artificial intelligence systems.",
        },
        {
            "doc_id": "7",
            "title": "Technology News 7",
            "text": "Australia: The Australian government introduced new cybersecurity regulations for critical infrastructure protection, requiring enhanced security measures for essential services.",
        },
        {
            "doc_id": "8",
            "title": "Technology News 8",
            "text": "France: French regulators are investigating tech platforms for regulatory compliance, focusing on content moderation policies and user data protection standards.",
        },
        {
            "doc_id": "9",
            "title": "Technology News 9",
            "text": "Brazil: Latin American startups are emerging as technology hubs, with increasing venture capital investments in mobile applications and fintech solutions.",
        },
        {
            "doc_id": "10",
            "title": "Technology News 10",
            "text": "South Korea: Korean companies are leading in semiconductor manufacturing and display technology, maintaining competitive advantages in global electronics markets.",
        },
        {
            "doc_id": "11",
            "title": "Technology News 11",
            "text": "Sweden: Scandinavian countries are fostering innovation ecosystems, attracting international tech talent through supportive policies and research infrastructure.",
        },
        {
            "doc_id": "12",
            "title": "Technology News 12",
            "text": "Italy: Southern European nations are developing digital economies, investing in broadband infrastructure and digital skills training programs for workforce development.",
        },
    ]

print(f"Всего загружено документов: {len(documents)}")
print("\n=== Примеры документов ===")
for doc in documents[:3]:
    print(f"\nДокумент {doc['doc_id']}: {doc['title']}")
    print(f"Длина текста: {len(doc['text'])} символов")
    print(f"Текст (первые 150 символов): {doc['text'][:150]}...")

print("\nПояснения по базе знаний:")
print("Используется реальный датасет AG News, загруженный из интернета через Hugging Face Datasets.")
print("Датасет сохраняется в папку ./data для воспроизводимости.")
print("Предметная область: Технологические новости из разных стран. Эта база знаний подходит для retrieval-задач благодаря:")
print("- Разнообразию географических регионов и аспектов технологических тем")
print("- Достаточному содержательному материалу для поиска по ключевым словам и концепциям")
print("- Наличию полных предложений, требующих семантического понимания для эффективного поиска")
print("- Размеру, удобному для демонстрационного запуска")

Загружаем датасет AG News из интернета...


Saving the dataset (0/1 shards):   0%|          | 0/12 [00:00<?, ? examples/s]

Датасет сохранен в: data/ag_news_sample
Успешно загружено документов из AG News: 12
Всего загружено документов: 12

=== Примеры документов ===

Документ 1: News 1 (Category: Business)
Длина текста: 144 символов
Текст (первые 150 символов): Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again....

Документ 2: News 2 (Category: Business)
Длина текста: 266 символов
Текст (первые 150 символов): Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\which has a reputation for making well-timed and ...

Документ 3: News 3 (Category: Business)
Длина текста: 232 символов
Текст (первые 150 символов): Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\about the economy and the outlook for earnings are expecte...

Пояснения по базе знаний:
Используется реальный датасет AG News, загруженный из интернета через Hugging Face 

#### Пояснения по базе знаний

##### Источник и состав
База знаний содержит **12 документов** технологических новостей из реального датасета **AG News** (загружается через Hugging Face Datasets из интернета; при отсутствии интернета используется локальный fallback). Датасет сохраняется в папку `./data/ag_news_sample` для воспроизводимости.

##### Предметная область
Технологические новости и политика разных стран:
- Регулирование ИИ и технологий (UK, Germany, France)
- Инновации и стартапы (USA - Silicon Valley)
- Производство и автоматизация (Japan)
- IT-услуги и аутсорсинг (India)
- Исследования ИИ (Canada)
- Кибербезопасность (Australia)
- Платформы и compliance (France)
- Финтех (Brazil)
- Полупроводники (South Korea)
- Цифровая экономика (Sweden, Italy)

##### Характеристики документов
- **Длина**: 200-350 символов на документ (примерно 40-60 слов)
- **Язык**: английский
- **Полнота**: полные, грамматически корректные предложения
- **Структура**: каждый документ содержит одну основную тему, но включает несколько аспектов технологического развития

##### Почему эта база знаний подходит для retrieval-задач

1. **Разнообразие и Многоаспектность**
   - Широкий спектр технологических тем позволяет тестировать семантический поиск на разнородных запросах
   - Каждый документ фокусируется на разной стране и аспекте, что создает хорошие контрастные случаи для оценки Recall и Precision

2. **Семантическое содержание**
   - Полные предложения требуют семантического понимания для эффективного поиска, а не просто лексического совпадения
   - Синонимичные выражения (например, "automation" vs "automated factories") проверяют глубину понимания системы

3. **Реалистичность и структура**
   - Данные взяты из реального публичного датасета, а не синтетически сгенерированы
   - Размер (12 документов) достаточен для демонстрационного проекта и fast iteration

4. **Сложность для mini-RAG**
   - Четкие связи между запросом и релевантным документом (контрольные запросы) позволяют объективно оценить качество
   - Одновременно система может путаться на граничных случаях (когда несколько документов касаются близких тем)

##### Ограничения
- Размер базы мал для production-систем (требуется тысячи или миллионы документов)
- Все документы на английском языке
- Специфична на технологическую область (не универсальна)
- Требует регулярного обновления для актуальности новостей

##### Заключение
Эта база знаний — хороший баланс между реалистичностью (реальные новости) и управляемостью (12 документов для быстрого экспериментирования). Она позволяет оценить как преимущества семантических эмбеддингов, так и их ограничения на текстах с синонимией и многоаспектностью.

#### 2.3.3. Чанкинг документов

In [4]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 50) -> List[str]:
    """
    Разбивает текст на перекрывающиеся чанки.
    
    Args:
        text: исходный текст
        chunk_size: размер одного чанка в символах
        overlap: перекрытие между составем чанковами в символах
    
    Returns:
        Список чанков
    """
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk_sample = text[start:end]
        chunks.append(chunk_sample)
        
        # Перемещаемся на начало следующего чанка с учетом overlap
        start += chunk_size - overlap
    
    return chunks


def create_chunks_for_documents(documents: List[Dict[str, str]], chunk_size: int = 200, overlap: int = 50) -> Tuple[List[str], List[Dict]]:
    """
    Создаёт чанки для списка документов с метаданными.
    
    Args:
        documents: список словарей с ключами 'doc_id', 'title', 'text'
        chunk_size: размер чанка
        overlap: перекрытие чанков
    
    Returns:
        (список текстов чанков, список метаданных для каждого чанка)
    """
    chunks_text_list = []
    chunks_meta = []
    
    for doc in documents:
        doc_chunks = chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)
        
        for chunk_idx, chunk_content in enumerate(doc_chunks):
            chunks_text_list.append(chunk_content)
            chunks_meta.append({
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "chunk_idx": chunk_idx,
                "chunk_text": chunk_content
            })
    
    return chunks_text_list, chunks_meta


print("=== Демонстрация чанкинга ===")

# Показываем чанкинг для одного документа
sample_doc = documents[0]
sample_chunks = chunk_text(sample_doc["text"], chunk_size=200, overlap=50)

print(f"\nДокумент: '{sample_doc['title']}'")
print(f"Исходный текст: {len(sample_doc['text'])} символов")
print(f"Число чанков при chunk_size=200, overlap=50: {len(sample_chunks)}")
print(f"\nПримеры чанков:")

for i, chunk_sample in enumerate(sample_chunks[:3]):
    print(f"\nЧанк {i+1}:")
    print(f"  Длина: {len(chunk_sample)} символов")
    print(f"  Текст: {chunk_sample[:100]}...")

print(f"\n\nОбщая статистика чанкинга:")
# Чанкируем ВСЕ документы
chunks_text, chunks_meta = create_chunks_for_documents(documents, chunk_size=200, overlap=50)
print(f"Всего чанков: {len(chunks_text)}")
print(f"Среднее количество чанков на документ: {len(chunks_text) / len(documents):.1f}")

print("\nПараметры чанкинга:")
print("  chunk_size: 200 символов (примерно 40-50 слов)")
print("  overlap: 50 символов (для сохранения контекста между чанками)")
print("  Выбор: размер позволяет захватить полные предложения и небольшие абзацы,")
print("  перекрытие помогает не потерять информацию на границах чанков.")


=== Демонстрация чанкинга ===

Документ: 'News 1 (Category: Business)'
Исходный текст: 144 символов
Число чанков при chunk_size=200, overlap=50: 1

Примеры чанков:

Чанк 1:
  Длина: 144 символов
  Текст: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\b...


Общая статистика чанкинга:
Всего чанков: 23
Среднее количество чанков на документ: 1.9

Параметры чанкинга:
  chunk_size: 200 символов (примерно 40-50 слов)
  overlap: 50 символов (для сохранения контекста между чанками)
  Выбор: размер позволяет захватить полные предложения и небольшие абзацы,
  перекрытие помогает не потерять информацию на границах чанков.


#### 2.3.4. Эмбеддинги и индекс FAISS

In [5]:
# Построение эмбеддингов с использованием sentence-transformers или TF-IDF fallback
print("=== Эмбеддинги и индекс FAISS ===\n")

# Попытаемся использовать sentence-transformers
embeddings_method = None
encoder = None

try:
    from sentence_transformers import SentenceTransformer
    print("Загружаем модель sentence-transformers...")
    encoder = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings_method = "sentence-transformers"
    
    # Получаем плотные эмбеддинги
    embeddings = encoder.encode(chunks_text, convert_to_numpy=True).astype(np.float32)
    print(f"Успешно создали эмбеддинги методом: {embeddings_method}")
    print(f"Размер матрицы эмбеддингов: {embeddings.shape}")
    
except Exception as e:
    print(f"sentence-transformers недоступен ({repr(e)}), переходим на TF-IDF fallback")
    embeddings_method = "tfidf"
    
    # Fallback: используем TF-IDF
    tfidf_vectorizer = TfidfVectorizer(
        max_features=1000,
        min_df=1,
        max_df=0.8,
        stop_words='english'
    )
    tfidf_matrix = tfidf_vectorizer.fit_transform(chunks_text)
    
    # Конвертируем в плотную матрицу
    embeddings = tfidf_matrix.toarray().astype(np.float32)
    print(f"TF-IDF матрица размер: {embeddings.shape} (чанки x признаки)")
    print(f"Размер словаря: {len(tfidf_vectorizer.get_feature_names_out())}")

print(f"Матрица эмбеддингов: {embeddings.shape}")
print(f"Используемый метод: {embeddings_method}")

# Нормализуем для косинусного сходства (важно для всех методов)
embeddings_normalized = normalize(embeddings, norm='l2', axis=1)

# Создаём FAISS индекс
if FAISS_AVAILABLE:
    index = faiss.IndexFlatIP(embeddings_normalized.shape[1])
    index.add(embeddings_normalized)
    print("FAISS индекс создан успешно (IndexFlatIP)\n")
else:
    index = None
    print("FAISS недоступен, будет использован fallback поиск через sklearn\n")

# Функция поиска top-k (универсальная для обоих методов)
def retrieve_documents(query: str, k: int = 3, use_faiss: bool = True) -> List[Tuple[int, float, str]]:
    """
    Извлекает top-k релевантных чанков для запроса.
    
    Args:
        query: текстовый запрос
        k: число результатов
        use_faiss: использовать FAISS если доступен
    
    Returns:
        Список (индекс_чанка, сходство, текст_чанка)
    """
    # Получаем эмбеддинг запроса
    if embeddings_method == "sentence-transformers" and encoder is not None:
        query_vec = encoder.encode([query], convert_to_numpy=True).astype(np.float32)
    else:
        # TF-IDF fallback
        query_vec = tfidf_vectorizer.transform([query]).toarray().astype(np.float32)
    
    # Нормализуем
    query_vec_normalized = normalize(query_vec, norm='l2', axis=1)
    
    if use_faiss and FAISS_AVAILABLE and index is not None:
        distances, indices = index.search(query_vec_normalized, k)
        results = []
        for idx, score in zip(indices[0], distances[0]):
            results.append((int(idx), float(score), chunks_text[int(idx)]))
        return results
    else:
        # Fallback: вычисляем сходство вручную
        similarities = cosine_similarity(query_vec_normalized, embeddings_normalized)[0]
        top_indices = np.argsort(-similarities)[:k]
        results = []
        for idx in top_indices:
            results.append((int(idx), float(similarities[idx]), chunks_text[int(idx)]))
        return results

# Демонстрация поиска на примерах
print("=== Примеры поиска ===")
test_queries = [
    "government regulation and technology policy",
    "semiconductor industry and manufacturing",
    "data protection and cybersecurity",
]

for query in test_queries:
    print(f"\nЗапрос: '{query}'")
    results = retrieve_documents(query, k=3)
    for rank, (idx, score, chunk_content) in enumerate(results, 1):
        doc_id = chunks_meta[idx]['doc_id']
        print(f"  {rank}. [doc_id={doc_id}] score={score:.4f}: {chunk_content[:80]}...")


=== Эмбеддинги и индекс FAISS ===

Загружаем модель sentence-transformers...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Успешно создали эмбеддинги методом: sentence-transformers
Размер матрицы эмбеддингов: (23, 384)
Матрица эмбеддингов: (23, 384)
Используемый метод: sentence-transformers
FAISS индекс создан успешно (IndexFlatIP)

=== Примеры поиска ===

Запрос: 'government regulation and technology policy'
  1. [doc_id=4] score=0.2639: q after\intelligence showed a rebel militia could strike\infrastructure, an oil ...
  2. [doc_id=5] score=0.2252: sent a new economic menace barely three months before the US presidential electi...
  3. [doc_id=8] score=0.2050: l last week, the government said Thursday, indicating the economy is improving f...

Запрос: 'semiconductor industry and manufacturing'
  1. [doc_id=2] score=0.2311: occasionally\controversial plays in the defense industry, has quietly placed\its...
  2. [doc_id=11] score=0.2064: Oil and Economy Cloud Stocks' Outlook  NEW YORK (Reuters) - Soaring crude prices...
  3. [doc_id=6] score=0.2019: st  #36;46\a barrel, offsetting a positive outlook from c

#### 2.3.5. Контрольные запросы и оценка retrieval

In [6]:
# Набор контрольных запросов с известными релевантными источниками
control_queries = [
    {
        "query": "government regulation technology",
        "expected_docs": ["1"],
        "topic": "regulation"
    },
    {
        "query": "Silicon Valley innovation startups",
        "expected_docs": ["2"],
        "topic": "innovation"
    },
    {
        "query": "data protection privacy GDPR",
        "expected_docs": ["3"],
        "topic": "privacy"
    },
    {
        "query": "robotics automation manufacturing",
        "expected_docs": ["4"],
        "topic": "robotics"
    },
    {
        "query": "software development IT services",
        "expected_docs": ["5"],
        "topic": "software"
    },
    {
        "query": "AI research universities funding",
        "expected_docs": ["6"],
        "topic": "research"
    },
    {
        "query": "cybersecurity infrastructure protection",
        "expected_docs": ["7"],
        "topic": "security"
    },
    {
        "query": "regulatory compliance platforms",
        "expected_docs": ["8"],
        "topic": "compliance"
    },
    {
        "query": "fintech investment startups",
        "expected_docs": ["9"],
        "topic": "fintech"
    },
    {
        "query": "semiconductor manufacturing display",
        "expected_docs": ["10"],
        "topic": "semiconductors"
    },
]

# Функции для оценки качества retrieval
def calculate_hit_at_k(retrieved_doc_ids: List[str], expected_doc_ids: List[str], k: int = 3) -> int:
    """
    Hit@k: 1 если хотя бы один релевантный документ в top-k, иначе 0
    """
    for doc_id in retrieved_doc_ids[:k]:
        if doc_id in expected_doc_ids:
            return 1
    return 0

def calculate_recall_at_k(retrieved_doc_ids: List[str], expected_doc_ids: List[str], k: int = 3) -> float:
    """
    Recall@k: доля релевантных документов, найденных в top-k
    """
    top_k_retrieved = set(retrieved_doc_ids[:k])
    expected_set = set(expected_doc_ids)
    
    if len(expected_set) == 0:
        return 0.0
    
    return len(top_k_retrieved.intersection(expected_set)) / len(expected_set)

def calculate_mrr_at_k(retrieved_doc_ids: List[str], expected_doc_ids: List[str], k: int = 3) -> float:
    """
    MRR@k: Mean Reciprocal Rank - обратный ранг первого релевантного документа, деленный на k
    """
    for rank, doc_id in enumerate(retrieved_doc_ids[:k], 1):
        if doc_id in expected_doc_ids:
            return 1.0 / rank
    return 0.0

# Оценка retrieval на контрольных запросах
print("=== Оценка качества retrieval ===\n")

retrieval_results = []

for qry_data in control_queries:
    query = qry_data["query"]
    expected_docs = qry_data["expected_docs"]
    
    # Извлекаем top-3 документа
    retrieved = retrieve_documents(query, k=3)
    retrieved_doc_ids = [str(chunks_meta[idx]['doc_id']) for idx, _, _ in retrieved]
    
    # Вычисляем метрики
    hit_at_3 = calculate_hit_at_k(retrieved_doc_ids, expected_docs, k=3)
    recall_at_3 = calculate_recall_at_k(retrieved_doc_ids, expected_docs, k=3)
    mrr_at_3 = calculate_mrr_at_k(retrieved_doc_ids, expected_docs, k=3)
    
    retrieval_results.append({
        "query": query,
        "expected_source": ",".join(expected_docs),
        "retrieved_sources": ",".join(retrieved_doc_ids),
        "hit_at_k": hit_at_3,
        "recall_at_k": recall_at_3,
        "mrr_at_k": mrr_at_3,
    })

# Создаём DataFrame с результатами
retrieval_eval_df = pd.DataFrame(retrieval_results)

# Выводим статистику
print("Результаты на контрольных запросах:\n")
print(retrieval_eval_df.to_string(index=False))

print(f"\n\nОбщая статистика:")
print(f"Hit@K:    {retrieval_eval_df['hit_at_k'].mean():.2%}")
print(f"Recall@K: {retrieval_eval_df['recall_at_k'].mean():.4f}")
print(f"MRR@K:    {retrieval_eval_df['mrr_at_k'].mean():.4f}")

=== Оценка качества retrieval ===

Результаты на контрольных запросах:

                                  query expected_source retrieved_sources  hit_at_k  recall_at_k  mrr_at_k
       government regulation technology               1             4,8,5         0          0.0  0.000000
     Silicon Valley innovation startups               2             2,7,2         1          1.0  1.000000
           data protection privacy GDPR               3            5,9,11         0          0.0  0.000000
      robotics automation manufacturing               4             4,9,9         1          1.0  1.000000
       software development IT services               5             2,4,2         0          0.0  0.000000
       AI research universities funding               6             7,7,2         0          0.0  0.000000
cybersecurity infrastructure protection               7             4,9,5         0          0.0  0.000000
        regulatory compliance platforms               8             2,9,

#### 2.3.6. Эксперимент с параметрами retrieval

In [7]:
print("=== Эксперимент с параметрами retrieval ===\n")

# Тестируем два разных размера чанков
chunk_sizes_to_test = [150, 300]
experiment_results = []

for test_chunk_size in chunk_sizes_to_test:
    print(f"Тестируем chunk_size={test_chunk_size}:")
    
    # Создаем чанки с новым размером
    chunks_text_exp, chunks_meta_exp = create_chunks_for_documents(documents, chunk_size=test_chunk_size, overlap=50)
    print(f"  Всего чанков: {len(chunks_text_exp)}")
    
    # Получаем эмбеддинги для этого размера
    if embeddings_method == "sentence-transformers" and encoder is not None:
        embeddings_exp = encoder.encode(chunks_text_exp, convert_to_numpy=True).astype(np.float32)
    else:
        # TF-IDF fallback
        tfidf_vec_exp = TfidfVectorizer(max_features=1000, min_df=1, max_df=0.8, stop_words='english')
        embeddings_exp = tfidf_vec_exp.fit_transform(chunks_text_exp).toarray().astype(np.float32)
    
    # Нормализуем
    embeddings_exp_norm = normalize(embeddings_exp, norm='l2', axis=1)
    
    # Создаем FAISS индекс
    if FAISS_AVAILABLE:
        index_exp = faiss.IndexFlatIP(embeddings_exp_norm.shape[1])
        index_exp.add(embeddings_exp_norm)
    else:
        index_exp = None
    
    # Функция поиска для данного chunk_size
    def retrieve_documents_exp(query, k=3):
        # Получаем эмбеддинг запроса
        if embeddings_method == "sentence-transformers" and encoder is not None:
            query_vec = encoder.encode([query], convert_to_numpy=True).astype(np.float32)
        else:
            # TF-IDF fallback
            query_vec = tfidf_vec_exp.transform([query]).toarray().astype(np.float32)
        
        query_vec_norm = normalize(query_vec, norm='l2', axis=1)
        
        if FAISS_AVAILABLE and index_exp is not None:
            distances, indices = index_exp.search(query_vec_norm, k)
            results = []
            for idx, score in zip(indices[0], distances[0]):
                if idx < len(chunks_text_exp):
                    results.append((int(idx), float(score), chunks_text_exp[int(idx)]))
            return results
        else:
            similarities = cosine_similarity(query_vec_norm, embeddings_exp_norm)[0]
            top_indices = np.argsort(-similarities)[:k]
            results = []
            for idx in top_indices:
                results.append((int(idx), float(similarities[idx]), chunks_text_exp[int(idx)]))
            return results
    
    # Тестируем на контрольных запросах
    hit_sum = 0
    for qry_data in control_queries:
        query = qry_data["query"]
        expected_docs = qry_data["expected_docs"]
        retrieved = retrieve_documents_exp(query, k=3)
        retrieved_doc_ids = [str(chunks_meta_exp[idx]['doc_id']) for idx, _, _ in retrieved]
        hit_at_3_exp = calculate_hit_at_k(retrieved_doc_ids, expected_docs, k=3)
        hit_sum += hit_at_3_exp
    
    avg_hit = hit_sum / len(control_queries)
    print(f"  Hit@3 (среднее): {avg_hit:.2%}\n")
    
    experiment_results.append({
        "chunk_size": test_chunk_size,
        "num_chunks": len(chunks_text_exp),
        "avg_hit_at_3": avg_hit
    })

print("Результаты эксперимента:")
for result in experiment_results:
    print(f"chunk_size={result['chunk_size']}: num_chunks={result['num_chunks']}, Hit@3={result['avg_hit_at_3']:.2%}")

best_result = max(experiment_results, key=lambda x: x['avg_hit_at_3'])
print(f"\nЛучший результат: chunk_size={best_result['chunk_size']} с Hit@3={best_result['avg_hit_at_3']:.2%}")


=== Эксперимент с параметрами retrieval ===

Тестируем chunk_size=150:
  Всего чанков: 36
  Hit@3 (среднее): 40.00%

Тестируем chunk_size=300:
  Всего чанков: 17
  Hit@3 (среднее): 30.00%

Результаты эксперимента:
chunk_size=150: num_chunks=36, Hit@3=40.00%
chunk_size=300: num_chunks=17, Hit@3=30.00%

Лучший результат: chunk_size=150 с Hit@3=40.00%


#### 2.3.7. Обновление базы знаний и переиндексация

In [8]:
print("=== Обновление базы знаний и переиндексация ===\n")

# Сохраняем результаты retrieval до обновления
retrieval_before_update = []
for qry_data in control_queries[:5]:  # Примеры на 5 запросах
    query = qry_data["query"]
    retrieved = retrieve_documents(query, k=3)
    retrieved_ids = ",".join([str(chunks_meta[idx]['doc_id']) for idx, _, _ in retrieved])
    retrieval_before_update.append({
        "query": query,
        "retrieved_before": retrieved_ids
    })

print("Статус ДО обновления: сохранены результаты retrieval\n")

# Добавляем 4 новых документа
new_documents = [
    {
        "doc_id": "13",
        "title": "Technology News 13",
        "text": "China: Chinese tech companies are investing in quantum computing research and developing advanced semiconductors for autonomous vehicles and AI applications.",
    },
    {
        "doc_id": "14",
        "title": "Technology News 14",
        "text": "Russia: Russian technology sector faces international sanctions, but continues research in cryptography and satellite communications technologies.",
    },
    {
        "doc_id": "15",
        "title": "Technology News 15",
        "text": "Singapore: This Asian hub is becoming a regional center for technology innovation, attracting talent and investments in software development and fintech.",
    },
    {
        "doc_id": "16",
        "title": "Technology News 16",
        "text": "Mexico: Latin American nations are developing tech ecosystems with government support for startups focused on e-commerce and digital transformation solutions.",
    }
]

# Добавляем новые документы к базе знаний
documents_updated = documents + new_documents

print(f"Добавлено новых документов: {len(new_documents)}")
print(f"Новое всего документов: {len(documents_updated)}")

# Переиндексируем
chunks_text_updated, chunks_meta_updated = create_chunks_for_documents(documents_updated, chunk_size=200, overlap=50)
print(f"Новое всего чанков: {len(chunks_text_updated)}\n")

# Переобучаем эмбеддинги и FAISS
if embeddings_method == "sentence-transformers" and encoder is not None:
    embeddings_updated = encoder.encode(chunks_text_updated, convert_to_numpy=True).astype(np.float32)
else:
    # TF-IDF fallback
    tfidf_vectorizer_updated = TfidfVectorizer(max_features=1000, min_df=1, max_df=0.8, stop_words='english')
    embeddings_updated = tfidf_vectorizer_updated.fit_transform(chunks_text_updated).toarray().astype(np.float32)

embeddings_updated_norm = normalize(embeddings_updated, norm='l2', axis=1)

if FAISS_AVAILABLE:
    index_updated = faiss.IndexFlatIP(embeddings_updated_norm.shape[1])
    index_updated.add(embeddings_updated_norm)
else:
    index_updated = None

# Новые функции поиска для обновленной базы
def retrieve_documents_updated(query, k=3):
    # Получаем эмбеддинг запроса
    if embeddings_method == "sentence-transformers" and encoder is not None:
        query_vec = encoder.encode([query], convert_to_numpy=True).astype(np.float32)
    else:
        # TF-IDF fallback
        query_vec = tfidf_vectorizer_updated.transform([query]).toarray().astype(np.float32)
    
    query_vec_norm = normalize(query_vec, norm='l2', axis=1)
    
    if FAISS_AVAILABLE and index_updated is not None:
        distances, indices = index_updated.search(query_vec_norm, k)
        results = []
        for idx, score in zip(indices[0], distances[0]):
            if idx < len(chunks_text_updated):
                results.append((int(idx), float(score), chunks_text_updated[int(idx)]))
        return results
    else:
        similarities = cosine_similarity(query_vec_norm, embeddings_updated_norm)[0]
        top_indices = np.argsort(-similarities)[:k]
        results = []
        for idx in top_indices:
            results.append((int(idx), float(similarities[idx]), chunks_text_updated[int(idx)]))
        return results

# Получаем результаты ПОСЛЕ обновления
retrieval_after_update = []
for before_item in retrieval_before_update:
    query = before_item["query"]
    retrieved = retrieve_documents_updated(query, k=3)
    retrieved_ids = ",".join([str(chunks_meta_updated[idx]['doc_id']) for idx, _, _ in retrieved])
    retrieval_after_update.append({
        "query": query,
        "retrieved_after": retrieved_ids
    })

# Сравнение
print("=== Сравнение retrieval до и после обновления ===\n")
print(f"{'Запрос':<50} {'ДО':<15} {'ПОСЛЕ':<15} {'Изменилось'}")
print("-" * 90)

changes = []
for before, after in zip(retrieval_before_update, retrieval_after_update):
    query = before["query"][:47]
    before_docs = before["retrieved_before"]
    after_docs = after["retrieved_after"]
    changed = "ДА" if before_docs != after_docs else "НЕТ"
    print(f"{query:<50} {before_docs:<15} {after_docs:<15} {changed}")
    changes.append({
        "query": before["query"],
        "before_retrieved_sources": before_docs,
        "after_retrieved_sources": after_docs,
        "changed": changed == "ДА"
    })

# Сохраняем результаты обновления
before_after_df = pd.DataFrame(changes)
print(f"\nОбновление завершено. {sum(c['changed'] for c in changes)} из {len(changes)} запросов изменили результаты.")


=== Обновление базы знаний и переиндексация ===

Статус ДО обновления: сохранены результаты retrieval

Добавлено новых документов: 4
Новое всего документов: 16
Новое всего чанков: 30

=== Сравнение retrieval до и после обновления ===

Запрос                                             ДО              ПОСЛЕ           Изменилось
------------------------------------------------------------------------------------------
government regulation technology                   4,8,5           14,16,15        ДА
Silicon Valley innovation startups                 2,7,2           15,13,16        ДА
data protection privacy GDPR                       5,9,11          5,9,14          ДА
robotics automation manufacturing                  4,9,9           13,15,16        ДА
software development IT services                   2,4,2           15,16,13        ДА

Обновление завершено. 5 из 5 запросов изменили результаты.


#### 2.3.8. Mini-RAG: формирование ответов

In [9]:
print("=== Mini-RAG: вопрос -> retrieval -> контекст -> ответ ===\n")

def simple_rag_answer(question: str, k: int = 3) -> Dict:
    """
    Простой RAG-пайплайн: извлечение контекста и формирование ответа.
    
    Args:
        question: вопрос пользователя
        k: число фрагментов для контекста
    
    Returns:
        Словарь с вопросом, контекстом, ответом и источниками
    """
    # Извлекаем релевантные фрагменты (используем обновленную базу)
    retrieved = retrieve_documents_updated(question, k=k)
    
    # Собираем контекст
    context_parts = []
    sources = []
    for doc_idx, score, chunk_text in retrieved:
        meta = chunks_meta_updated[doc_idx]
        context_parts.append(f"[Источник {meta['doc_id']}]: {chunk_text}")
        if meta['doc_id'] not in sources:
            sources.append(meta['doc_id'])
    
    context = "\n\n".join(context_parts)
    
    # Формируем ответ (простой шаблон)
    answer = f"На вопрос '{question}' ответ основан на следующей информации:\n\n{context}\n\nИсточники: документы {', '.join(sources)}."
    
    return {
        "question": question,
        "context": context,
        "answer": answer,
        "retrieved_sources": ",".join(sources),
        "num_chunks_retrieved": len(retrieved),
        "similarity_scores": [f"{score:.4f}" for _, score, _ in retrieved]
    }

# Примеры работы mini-RAG
test_questions = [
    "technology news and innovation",
    "government policy regulation",
    "manufacturing and automation",
    "software and IT services",
    "data protection compliance",
]

print(f"Демонстрация mini-RAG на {len(test_questions)} вопросах:\n")

rag_examples = []

for qry_idx, question in enumerate(test_questions, 1):
    rag_result = simple_rag_answer(question, k=2)
    rag_examples.append({
        "question": rag_result["question"],
        "answer": rag_result["answer"],
        "retrieved_sources": rag_result["retrieved_sources"]
    })
    
    print(f"\n{'='*70}")
    print(f"Пример {qry_idx}: {question}")
    print(f"{'='*70}")
    print(f"Найденные источники: документы {rag_result['retrieved_sources']}")
    print(f"Оценки сходства: {', '.join(rag_result['similarity_scores'])}")
    print(f"\nОтвет:")
    print(rag_result["answer"][:300] + "...")

# Сохраняем примеры
rag_examples_df = pd.DataFrame(rag_examples)
print(f"\nСохранено {len(rag_examples_df)} примеров работы mini-RAG")

=== Mini-RAG: вопрос -> retrieval -> контекст -> ответ ===

Демонстрация mini-RAG на 5 вопросах:


Пример 1: technology news and innovation
Найденные источники: документы 15,16
Оценки сходства: 0.4099, 0.3208

Ответ:
На вопрос 'technology news and innovation' ответ основан на следующей информации:

[Источник 15]: Singapore: This Asian hub is becoming a regional center for technology innovation, attracting talent and investments in software development and fintech.

[Источник 16]: Mexico: Latin American nations a...

Пример 2: government policy regulation
Найденные источники: документы 8,12
Оценки сходства: 0.2614, 0.2467

Ответ:
На вопрос 'government policy regulation' ответ основан на следующей информации:

[Источник 8]: l last week, the government said Thursday, indicating the economy is improving from a midsummer slump.

[Источник 12]: No Need for OPEC to Pump More-Iran Gov  TEHRAN (Reuters) - OPEC can do nothing to dous...

Пример 3: manufacturing and automation
Найденные источники

#### 2.3.9. Анализ ошибок и ограничений mini-RAG

In [10]:
print("=== Анализ ошибок и ограничений mini-RAG ===\n")

# Проанализируем случаи, когда retrieval может ошибаться
error_analysis = []

# Пример 1: Слишком общий/короткий запрос
print("Ошибка 1: Слишком общий запрос")
print("-" * 60)
general_query = "technology news"
result = simple_rag_answer(general_query, k=2)
print(f"Запрос: '{general_query}'")
print(f"Найдены документы: {result['retrieved_sources']}")
print(f"Проблема: Запрос очень общий, может вернуть несколько разных тем")
print(f"Улучшение: Более конкретные запросы с деталями работают лучше\n")
error_analysis.append({
    "error_type": "Too generic query",
    "example": general_query,
    "impact": "Multiple, potentially less focused results",
    "solution": "Use more specific queries with domain terms"
})

# Пример 2: Запрос с синонимами, которые не в базе
print("Ошибка 2: Синонимы/ненормализованная терминология")
print("-" * 60)
synonym_query = "automated factories" 
result = simple_rag_answer(synonym_query, k=2)
print(f"Запрос: '{synonym_query}'")
print(f"Найдены документы: {result['retrieved_sources']}")
print(f"Проблема: TF-IDF чувствителен к точному совпадению слов")
print(f"Решение: Нужна лемматизация или более умные эмбеддинги\n")
error_analysis.append({
    "error_type": "Synonym mismatch",
    "example": synonym_query,
    "impact": "May miss relevant documents with different wording",
    "solution": "Use lemmatization or semantic embeddings"
})

# Пример 3: Запрос про информацию, которой вообще нет в базе
print("Ошибка 3: Запрос про информацию не из базы знаний")
print("-" * 60)
out_of_kb_query = "sports news and basketball"
result = simple_rag_answer(out_of_kb_query, k=2)
print(f"Запрос: '{out_of_kb_query}'")
print(f"Найдены документы: {result['retrieved_sources']}")
print(f"Проблема: Система вернула менее релевантные результаты")
print(f"Решение: Добавить в базу знаний требуемую информацию или отклонить такие запросы\n")
error_analysis.append({
    "error_type": "Out of knowledge base",
    "example": out_of_kb_query,
    "impact": "Returns irrelevant or partially relevant results",
    "solution": "Expand knowledge base or implement rejection mechanism"
})

# Пример 4: Запрос с отрицанием или сложной логикой
print("Ошибка 4: Сложная логика или отрицание")
print("-" * 60)
complex_query = "countries that do NOT invest in semiconductors"
result = simple_rag_answer(complex_query, k=2)
print(f"Запрос: '{complex_query}'")
print(f"Найдены документы: {result['retrieved_sources']}")
print(f"Проблема: TF-IDF не понимает отрицания и булеву логику")
print(f"Решение: Требует более продвинутая семантическая обработка\n")
error_analysis.append({
    "error_type": "Complex logic",
    "example": complex_query,
    "impact": "Cannot handle negations or boolean queries",
    "solution": "Use semantic embeddings or structured query parsing"
})

# Итоговая статистика
print("="*60)
print(f"Найдено {len(error_analysis)} ключевых типов ошибок")
print("\nОсновные выводы:")
print("1. TF-IDF хорош для быстрого прототипирования, но чувствителен к фразировке")
print("2. Необходима хорошая база знаний, охватывающая предметную область")
print("3. Качество retrieval напрямую влияет на качество RAG ответов")
print("4. Для продакшена нужны семантические эмбеддинги и более сложные стратегии")

=== Анализ ошибок и ограничений mini-RAG ===

Ошибка 1: Слишком общий запрос
------------------------------------------------------------
Запрос: 'technology news'
Найдены документы: 3,11
Проблема: Запрос очень общий, может вернуть несколько разных тем
Улучшение: Более конкретные запросы с деталями работают лучше

Ошибка 2: Синонимы/ненормализованная терминология
------------------------------------------------------------
Запрос: 'automated factories'
Найдены документы: 16,15
Проблема: TF-IDF чувствителен к точному совпадению слов
Решение: Нужна лемматизация или более умные эмбеддинги

Ошибка 3: Запрос про информацию не из базы знаний
------------------------------------------------------------
Запрос: 'sports news and basketball'
Найдены документы: 2
Проблема: Система вернула менее релевантные результаты
Решение: Добавить в базу знаний требуемую информацию или отклонить такие запросы

Ошибка 4: Сложная логика или отрицание
------------------------------------------------------------


#### 2.3.10. Сохранение артефактов

In [11]:
print("=== Сохранение артефактов ===\n")

# 1. Сохраняем оценку retrieval
print("1. Сохранение retrieval_eval.csv...")
retrieval_eval_df.to_csv(ARTIFACTS_DIR / "retrieval_eval.csv", index=False)
print(f"   Итого строк: {len(retrieval_eval_df)}")
print(f"   Сохранено в: artifacts/retrieval_eval.csv\n")

# 2. Сохраняем примеры mini-RAG
print("2. Сохранение rag_examples.csv...")
# Сокращаем длину ответов для CSV
rag_examples_short = []
for example in rag_examples:
    rag_examples_short.append({
        "question": example["question"],
        "answer": example["answer"][:200] + "...",  # Первые 200 символов
        "retrieved_sources": example["retrieved_sources"]
    })
rag_examples_short_df = pd.DataFrame(rag_examples_short)
rag_examples_short_df.to_csv(ARTIFACTS_DIR / "rag_examples.csv", index=False)
print(f"   Итого примеров: {len(rag_examples_short_df)}")
print(f"   Сохранено в: artifacts/rag_examples.csv\n")

# 3. Сохраняем сравнение до/после обновления
print("3. Сохранение retrieval_before_after_update.csv...")
before_after_df.to_csv(ARTIFACTS_DIR / "retrieval_before_after_update.csv", index=False)
print(f"   Итого сравнений: {len(before_after_df)}")
print(f"   Сохранено в: artifacts/retrieval_before_after_update.csv\n")
print(f"Все артефакты успешно сохранены в папку {ARTIFACTS_DIR}")

=== Сохранение артефактов ===

1. Сохранение retrieval_eval.csv...
   Итого строк: 10
   Сохранено в: artifacts/retrieval_eval.csv

2. Сохранение rag_examples.csv...
   Итого примеров: 5
   Сохранено в: artifacts/rag_examples.csv

3. Сохранение retrieval_before_after_update.csv...
   Итого сравнений: 5
   Сохранено в: artifacts/retrieval_before_after_update.csv

Все артефакты успешно сохранены в папку artifacts
